# 0910 9일차

## 0. 파이썬 문법 - 판다스를 넘파이로 바꾸기

산탄데르에서 y가 `Series`(판다스)로 나옴

**변환 방법**
1. `np.array(y)` : 넘파이 함수로 감싸기 → 리스트·튜플에도 씀
2. `y.to_numpy()` : 판다스 메서드 (괄호 O) → 판다스가 권장하는 방법
3. `y.values` : 판다스 속성 (괄호 X) → 8일차에서 원핫 뒤에 붙이던 것

```python
y = np.array(y)     # 방법 1 - 넘파이 함수로 감싸기
y = y.to_numpy()    # 방법 2 - 판다스 메서드
y = y.values        # 방법 3 - 판다스 속성
```

- `.values`에 괄호가 없는 것은 속성이라 호출하는 것이 아니기 때문임 (7일차 §0 속성 접근, 8일차 §0-2 메서드 체이닝)

### 0-1. 넘파이로 바꾸는 이유

```python
y = pd.get_dummies(y).values    # get_dummies는 DataFrame을 돌려주므로 .values로 꺼냄
```

- 원핫 결과가 판다스면 `np.argmax(y_test, axis=1)` 같은 넘파이 연산에서 축·인덱스가 헷갈림
- 케라스도 내부에서는 넘파이로 바꿔 쓰므로 처음부터 넘파이로 통일하는 것이 안전함

**cf) get_dummies에 dtype=float을 빠뜨린 경우**

```python
y = pd.get_dummies(y).values                # keras24 - dtype 지정 안 함 -> bool
y = pd.get_dummies(y, dtype=float).values   # keras26, 28 - float
```

- 기본 dtype이 `bool`임 (8일차 §3-1)
- 텐서플로가 캐스팅해줘서 실행은 되지만, `dtype=float`을 붙이는 것이 좋음

## 1. 이진분류를 다중분류로 풀기 (산탄데르)

산탄데르는 y가 0/1 두 개라 원래는 이진분류(7일차)이지만, 라벨 2개짜리 다중분류로 바꿔서 풀 수도 있음 (`keras24`)

| | 이진으로 (7일차) | 다중으로 (오늘) |
|---|---|---|
| y 가공 | 없음 `(n,)` | 원핫 `(n, 2)` |
| 출력층 | `Dense(1, activation='sigmoid')` | `Dense(2, activation='softmax')` |
| loss | `binary_crossentropy` | `categorical_crossentropy` |
| 예측 후처리 | `np.round` | `np.argmax` |

- 라벨이 2개면 두 방법 모두 됨 → 이진분류는 다중분류의 특수한 경우라고 볼 수 있음
- 보통은 이진 쪽이 간단함. 노드가 1개라 파라미터가 적고 원핫도 필요 없음

### 1-1. 제출할 확률 열 고르기 (`[:, 1]`)

- sigmoid는 값이 하나라 그대로 내면 되지만, softmax는 `(n, 2)`라서 어느 열을 낼지 정해야 함

```python
y_submit = model.predict(test)      # (200000, 2)  [[0.97, 0.03], [0.88, 0.12], ...]
submit['target'] = y_submit[:, 1]   # 1번(양성) 열만
```

```
    0열      1열
[[ 0.97 ,  0.03 ]]     <- 0일 확률 / 1일 확률
           ^^^^
           이 열이 제출값
```

**규칙**
1. 대회가 요구하는 것은 "target이 1일 확률"이라 1번 열을 냄
2. 여기서 `argmax`를 쓰면 0 또는 1만 남아 확률 정보가 사라짐
   - 채점용은 argmax, 제출용은 확률 그대로

### 1-2. 불균형 데이터에서 accuracy의 한계

```
acc_score :  0.91105
time :  300.07 sec
```

라벨 분포

```
0 : 179902
1 :  20098      <- 10%
```

**결과 해석**
- 전부 0이라고만 찍어도 정확도가 0.89951임 (베이스라인)
- 0.911은 베이스라인보다 1.2%p 높은 값 (학습 300초)
- `stratify=y`로 비율은 지켰지만 불균형 자체가 사라지지는 않음 (7일차 §3-3)
- 이런 데이터에서는 accuracy로 성능을 판단하기 어려움 (산탄데르 대회의 실제 평가지표도 AUC)

## 2. summary()와 파라미터 개수

모델의 층 구성과 학습할 w, b의 개수를 표로 보여주는 메서드 (`keras25`)

```python
model.summary()
```

```
┌─────────────────┬──────────────┬───────────┐
│ Layer (type)    │ Output Shape │   Param # │
├─────────────────┼──────────────┼───────────┤
│ dense (Dense)   │ (None, 3)    │         6 │
│ dense_1 (Dense) │ (None, 4)    │        16 │
│ dense_2 (Dense) │ (None, 3)    │        15 │
│ dense_3 (Dense) │ (None, 1)    │         4 │
└─────────────────┴──────────────┴───────────┘
 Total params: 41
```

### 2-1. Dense 파라미터 계산 공식

$$\text{Param} = (\text{입력 노드 수} + 1) \times \text{출력 노드 수}$$

- 입력 노드 수 × 출력 노드 수 = 가중치 w의 개수
- +1은 bias → 출력 노드마다 b가 하나씩 붙음
- 1일차 §3의 `y = wx + b`가 노드 하나마다 하나씩 있는 것임

| 층 | 입력 | 출력 | 계산 | Param |
|---|---|---|---|---|
| `Dense(3, input_dim=1)` | 1 | 3 | (1+1) × 3 | 6 |
| `Dense(4)` | 3 | 4 | (3+1) × 4 | 16 |
| `Dense(3)` | 4 | 3 | (4+1) × 3 | 15 |
| `Dense(1)` | 3 | 1 | (3+1) × 1 | 4 |
| | | | | **41** |

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(3, input_dim=1))    # (1+1) * 3 = 6
model.add(Dense(4))                 # (3+1) * 4 = 16
model.add(Dense(3))                 # (4+1) * 3 = 15
model.add(Dense(1))                 # (3+1) * 1 = 4

model.summary()                     # Total params: 41

## 3. input_shape

입력 데이터의 모양을 튜플로 지정하는 인자

```python
model.add(Dense(10, input_dim=4, activation='relu'))        # 지금까지
model.add(Dense(10, input_shape=(4,), activation='relu'))   # 오늘
```

**input_dim과의 차이**
- 둘은 같은 뜻임. iris는 열이 4개라 `4` 또는 `(4,)`
- `input_dim`은 숫자 하나라서 1차원(열 개수)밖에 표현하지 못함
- `input_shape`는 튜플이라 이미지처럼 3차원 이상도 표현할 수 있음

| | 쓸 수 있는 데이터 |
|---|---|
| `input_dim=4` | 표 형태(2차원)만 |
| `input_shape=(4,)` | 표 형태 |
| `input_shape=(100, 100, 3)` | 이미지처럼 3차원 이상 |

### 3-1. input_shape 규칙 - 행 개수를 뺀 튜플

```
data shape           input shape
(n, 4)             →  (4,)
(n, 100, 3)        →  (100, 3)
(n, 100, 100, 3)   →  (100, 100, 3)
```

- 행 개수 `n`은 데이터가 몇 개인지일 뿐 모델 구조와 무관해서 뺌
- `summary()`의 `Output Shape`가 `(None, 3)`인 것도 같은 이유 → `None`이 그 `n` 자리임
- 이미지를 다루면 `(가로, 세로, 채널)`이 되므로 `input_shape`만 쓸 수 있음

## 4. 스케일링 (Scaling)

특성마다 다른 값의 범위를 비슷한 범위로 맞추는 전처리

**스케일링이 필요한 이유**

캘리포니아 주택 데이터의 실제 값 범위 (`keras27`)

```
MedInc          0.500 ~        15.000      <- 소득
HouseAge        1.000 ~        52.000
AveRooms        0.846 ~       141.909
AveBedrms       0.333 ~        34.067
Population      3.000 ~     35682.000      <- 인구
AveOccup        0.692 ~      1243.333
Latitude       32.540 ~        41.950
Longitude    -124.350 ~      -114.310      <- 음수
```

- 소득은 최대 15인데 인구는 35,682로 2천 배 차이가 남
- `w`를 갱신할 때 숫자가 큰 특성이 loss를 지배함
  - 인구는 조금만 움직여도 loss가 크게 변하고, 소득은 아무리 움직여도 영향이 작음
- 3일차 §5-3에서 정리한 "특성 스케일이 제각각이면 생기는 문제"의 해결책임

### 4-1. MinMaxScaler

모든 값을 0~1 범위로 바꾸는 스케일러

$$x' = \frac{x - \text{MIN}}{\text{MAX} - \text{MIN}}$$

**공식의 의미**
1. `x - MIN` : 시작점을 0으로 맞춤
   - 가장 큰 수로 나누기만 하면 최대값은 1이 되지만 최소값이 0이 되지 않음
2. `/ (MAX - MIN)` : 폭으로 나눠 끝을 1로 맞춤

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(x)           # MIN, MAX를 계산 (기준을 잡음)
x = scaler.transform(x) # 그 기준으로 변환
```

**메서드의 역할**
1. `fit` : 기준(MIN, MAX)을 정하는 단계
2. `transform` : 정한 기준을 적용하는 단계 (이 구분이 §5에서 중요해짐)

- 모든 특성이 0~1이 되면 어느 것도 크기만으로 우위를 갖지 못함

### 4-2. 스케일링 효과

`keras27`은 층 구성·epoch를 바꾸지 않고 스케일링만 넣음

| | 스케일링 전 | 후 |
|---|---|---|
| loss (mse) | 0.9900 | 0.5106 |
| R² | 0.2542 | 0.6154 |
| RMSE | 0.9950 | 0.7146 |

- R²가 약 2.4배 올라감

**cf) `1.0000000000000002`는 부동소수점 오차**

```python
print(np.min(x), np.max(x))     # 0.0 1.0000000000000002
```

- 1을 넘은 것이 아니라 `(MAX - MIN) / (MAX - MIN)`이 정확히 1로 떨어지지 않은 것
- 10진수 소수를 2진수로 저장하면서 생기는 미세한 차이라 무시해도 됨

## 5. 스케일링 순서

**규칙**
1. train / test를 먼저 분리함
2. `fit`은 train에만 함
3. test는 train 기준으로 `transform`만 함

```python
# X - 잘못된 순서 (keras27)
scaler.fit(x)                                       # 전체로 기준을 잡음
x = scaler.transform(x)
x_train, x_test, ... = train_test_split(x, y, ...)  # 그 다음에 분리
```

```python
# O - 올바른 순서 (keras28)
x_train, x_test, ... = train_test_split(x, y, ...)  # 1) 먼저 분리

scaler.fit(x_train)                                 # 2) train으로만 기준을 잡고
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)                   # 3) test는 그 기준으로 변환만
```

### 5-1. 전체로 fit하면 안 되는 이유 - 데이터 누수

```
전체로 fit    :  scaler가 본 것 = train + test    <- test 정보가 섞임
train으로 fit :  scaler가 본 것 = train만         <- 정상
```

- 전체로 `fit`하면 scaler가 test의 MIN/MAX까지 알게 됨 → test 정보가 훈련 과정에 들어감
- 5일차 §2-3의 `casual + registered = count`와 같은 종류의 문제임
- 결과적으로 평가 점수가 실제 성능보다 좋게 나옴
- 실전에서는 아직 존재하지 않는 데이터가 들어오므로, 그 MIN/MAX를 미리 알 방법이 없음

### 5-2. x_test가 0~1을 벗어나는 이유

train 기준으로만 변환하므로, train에서 보지 못한 범위의 값은 0~1 밖으로 나감

```python
print(np.min(x_train), np.max(x_train))     # 0.0 1.0000000000000002
print(np.min(x_test),  np.max(x_test))      # -0.005005005005005 2.0745532963647566
```

max가 2.07인 이유

```
AveOccup 컬럼
  train 의 MIN / MAX  :    0.75 /  599.71
  그 test 행의 원본 값 : 1243.33          <- train 에서 본 적 없는 극단값

  (1243.33 - 0.75) / (599.71 - 0.75) = 2.0746
```

| | 개수 |
|---|---|
| 1 초과 | 1 / 33,024 |
| 0 미만 | 2 / 33,024 |

- 33,024개 중 3개뿐이고, 이 3개가 train에서 보지 못한 범위의 값임
- 이것은 정상임. x_test를 다시 `fit`하면 2.07이 1.0으로 바뀌면서 이상치라는 사실이 사라짐

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

d = fetch_california_housing()
x, y = d.data, d.target

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=121)

scaler = MinMaxScaler()
scaler.fit(x_train)                     # train 으로만 fit
x_train = scaler.transform(x_train)
x_test = scaler.transform(x_test)       # test 는 transform 만

print(np.min(x_train), np.max(x_train)) # 0.0 1.0000000000000002
print(np.min(x_test),  np.max(x_test))  # -0.005005005005005 2.0745532963647566

# 2.07 이 어느 컬럼인지 찾아보기
i, j = np.unravel_index(np.argmax(x_test), x_test.shape)
print(d.feature_names[j])                           # AveOccup
print(scaler.data_min_[j], scaler.data_max_[j])     # 0.75 599.7142857142857

### 5-3. cf) validation_split에 남아 있는 누수

```python
scaler.fit(x_train)                                     # x_train 전체로 fit
model.fit(x_train, y_train, validation_split=0.2)       # 그 x_train에서 val을 떼감
```

- val도 결국 x_train에서 잘라내므로, val의 MIN/MAX도 scaler가 이미 알고 있음
- 엄밀히 하려면 val을 먼저 떼고(6일차 §3-3의 `validation_data`) train으로만 `fit`해야 함
- 실무에서도 보통 이 정도는 넘어가지만, 누수가 없는 것은 아님